# Clase 20 · Speaker verification con ECAPA-TDNN

**Recorrido:** cargar cuatro WAV → obtener cuatro embeddings → comparar tres pares.
¿Cómo cambia la semejanza cuando varían la persona, las palabras o el ruido?

Selecciona **UNIE_VozClon**, reinicia el kernel y ejecuta las celdas en orden.
Usa los archivos `A1_clara.wav`, `A2_otra_frase.wav`, `A1_ruido.wav` y
`B1_otra_persona.wav` de `data/`. Este notebook lee esos archivos y no activa el micrófono.


In [1]:
import os
os.environ["MKL_THREADING_LAYER"] = "SEQUENTIAL"

from pathlib import Path
import numpy as np
import torch
import torchaudio
from IPython.display import Audio, display


Comparamos cuatro grabaciones conocidas: dos de la persona A con frases distintas,
una grabación de A con ruido y una de otra persona. Los archivos se cargan por
rutas relativas fijas desde `data/`; no se eligen por índices de una lista.

| Archivo | Condición |
|---|---|
| `A1_clara.wav` | Persona A, voz clara |
| `A2_otra_frase.wav` | Persona A, otra frase |
| `A1_ruido.wav` | Persona A, con ruido |
| `B1_otra_persona.wav` | Persona B |

## 1. Cargar y escuchar los archivos


In [2]:
rutas_voz = {
    "A1_clara": Path("data/A1_clara.wav"),
    "A2_otra_frase": Path("data/A2_otra_frase.wav"),
    "A1_ruido": Path("data/A1_ruido.wav"),
    "B1_otra_persona": Path("data/B1_otra_persona.wav"),
}

faltantes = [str(ruta) for ruta in rutas_voz.values() if not ruta.is_file()]
if faltantes:
    raise FileNotFoundError("Faltan grabaciones requeridas: " + ", ".join(faltantes))
for nombre, ruta in rutas_voz.items():
    print(f"{nombre}: {ruta}")


A1_clara: data\A1_clara.wav
A2_otra_frase: data\A2_otra_frase.wav
A1_ruido: data\A1_ruido.wav
B1_otra_persona: data\B1_otra_persona.wav


Leemos cada WAV, convertimos las muestras a `float32` mono y remuestreamos a
**16 kHz** si hace falta. Se reproducen las cuatro muestras para verificar que
corresponden a las condiciones indicadas.


In [3]:
from scipy.io import wavfile

audios_voz = {}
for nombre, ruta in rutas_voz.items():
    frecuencia, muestras = wavfile.read(ruta)
    if muestras.dtype == np.uint8:
        muestras = (muestras.astype(np.float32) - 128) / 128
    elif np.issubdtype(muestras.dtype, np.integer):
        escala = float(-np.iinfo(muestras.dtype).min)
        muestras = muestras.astype(np.float32) / escala
    else:
        muestras = muestras.astype(np.float32)
    if muestras.ndim == 2:
        muestras = muestras.mean(axis=1)
    if muestras.ndim != 1 or muestras.size == 0 or not np.isfinite(muestras).all():
        raise ValueError(f"Revisa el audio {nombre}: contiene muestras no válidas.")
    if np.max(np.abs(muestras)) == 0:
        raise ValueError(f"El audio {nombre} contiene solo ceros; vuelve a grabarlo.")
    senal = torch.from_numpy(np.ascontiguousarray(muestras)).unsqueeze(0)
    if frecuencia != 16000:
        senal = torchaudio.functional.resample(senal, frecuencia, 16000)
    audios_voz[nombre] = senal
    print(f"{nombre}: {ruta.name} | {senal.shape[1] / 16000:.3f} s")
    display(Audio(senal[0].numpy(), rate=16000, normalize=False))


A1_clara: A1_clara.wav | 6.000 s


A2_otra_frase: A2_otra_frase.wav | 6.000 s


A1_ruido: A1_ruido.wav | 6.000 s


B1_otra_persona: B1_otra_persona.wav | 6.000 s


## 2. Cargar el modelo preentrenado

**Descarga de pesos y configuración:** la primera ejecución obtiene
[ECAPA-TDNN de SpeechBrain](https://huggingface.co/speechbrain/spkrec-ecapa-voxceleb)
desde Hugging Face y lo conserva en `data/modelos/ecapa_tdnn/`.
Después reutiliza los archivos. La inferencia se realiza localmente en CPU,
con pesos fijos. Cada grabación produce un embedding de **192 componentes**.

Instala las dependencias del proyecto con `python -m pip install -r requirements.txt`.


In [4]:
from speechbrain.inference.classifiers import EncoderClassifier
from speechbrain.utils.fetching import LocalStrategy

encoder_voz = EncoderClassifier.from_hparams(
    source="speechbrain/spkrec-ecapa-voxceleb",
    savedir="data/modelos/ecapa_tdnn",
    run_opts={"device": "cpu"},
    local_strategy=LocalStrategy.COPY_SKIP_CACHE
)
encoder_voz.eval()
encoder_voz.requires_grad_(False)
print("ECAPA-TDNN cargado.")

c:\Users\santi\miniconda3\envs\UNIE_VozClon\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\santi\miniconda3\envs\UNIE_VozClon\Lib\site-packages\speechbrain\utils\torch_audio_backend.py:57: UserWarning: torchaudio._backend.list_audio_backends has been deprecated. This deprecation is part of a large refactoring effort to transition TorchAudio into a maintenance phase. The decoding and encoding capabilities of PyTorch for both audio and video are being consolidated into TorchCodec. Please see https://github.com/pytorch/audio/issues/3902 for more information. It will be removed from the 2.9 release. 
  available_backends = torchaudio.list_audio_backends()
c:\Users\santi\miniconda3\envs\UNIE_VozClon\Lib\site-packages\huggingface_hub\file_download.py:943: FutureWarning: The `force_filename` parameter is deprec

ECAPA-TDNN cargado.


## 3. Extraer embeddings y comparar

El encoder procesa cada grabación por separado, aunque sus longitudes sean distintas.
Normalizamos los embeddings a longitud uno; su producto escalar será la
**cosine similarity**, un score entre −1 y 1.

In [5]:
nombres_voz = list(audios_voz)
vectores_voz = []
with torch.inference_mode():
    for nombre in nombres_voz:
        embedding = encoder_voz.encode_batch(audios_voz[nombre]).reshape(-1)
        vectores_voz.append(embedding)

embeddings_voz = torch.stack(vectores_voz)
embeddings_voz = torch.nn.functional.normalize(embeddings_voz, dim=1)
scores_voz = embeddings_voz @ embeddings_voz.T
indice_voz = {nombre: indice for indice, nombre in enumerate(nombres_voz)}
print(f"Embeddings (grabaciones, componentes): {tuple(embeddings_voz.shape)}")


Embeddings (grabaciones, componentes): (4, 192)


In [6]:
from IPython.display import Markdown

pares_voz = [
    ("A1_clara", "A2_otra_frase", "Misma persona, frases distintas"),
    ("A1_clara", "A1_ruido", "Misma persona, una toma con ruido"),
    ("A1_clara", "B1_otra_persona", "Personas distintas"),
]
filas = ["| Par | Condición | Similitud coseno |", "|---|---|---:|"]
for primero, segundo, condicion in pares_voz:
    score = scores_voz[indice_voz[primero], indice_voz[segundo]].item()
    print(f"{primero}↔{segundo}: {score:.4f}")
    filas.append(f"| {primero} ↔ {segundo} | {condicion} | {score:.4f} |")
display(Markdown("\n".join(filas)))


A1_clara↔A2_otra_frase: 0.8493
A1_clara↔A1_ruido: 0.8555
A1_clara↔B1_otra_persona: 0.2762


| Par | Condición | Similitud coseno |
|---|---|---:|
| A1_clara ↔ A2_otra_frase | Misma persona, frases distintas | 0.8493 |
| A1_clara ↔ A1_ruido | Misma persona, una toma con ruido | 0.8555 |
| A1_clara ↔ B1_otra_persona | Personas distintas | 0.2762 |

Comprueba si las dos grabaciones de la persona A se parecen más entre sí que
`A1_clara.wav` y `B1_otra_persona.wav`. El score es semejanza entre embeddings,
**no una probabilidad de identidad**; para tomar una decisión haría falta validar
un umbral con más voces y condiciones.


**Modelo:** [ECAPA-TDNN de SpeechBrain](https://huggingface.co/speechbrain/spkrec-ecapa-voxceleb), licencia Apache 2.0.
Las dependencias, incluidas SpeechBrain 1.1.1 y Hugging Face Hub 0.36.0, se mantienen en `requirements.txt`.
La primera celda configura MKL en modo secuencial para evitar el conflicto OpenMP de UNIE_VozClon.


## Conclusiones de verificación de locutor

| Par | Condición | Similitud coseno |
|---|---|---:|
| `A1_clara` ↔ `A2_otra_frase` | Misma persona, frases distintas | 0.8493 |
| `A1_clara` ↔ `A1_ruido` | Misma persona, una toma con ruido | 0.8555 |
| `A1_clara` ↔ `B1_otra_persona` | Personas distintas | 0.2762 |

ECAPA-TDNN transforma cada señal en un embedding que resume características
acústicas del hablante (por ejemplo, patrones espectrales y de voz) y las entrena
para que las muestras de una misma identidad queden próximas en el espacio de
representación. Al no modelar la secuencia de palabras como objetivo principal,
puede conservar similitud entre frases distintas; la pronunciación y el contenido
siguen pudiendo influir en el resultado.

El ruido ambiental puede enmascarar rasgos de la voz y degradar el embedding, por
lo que la similitud de `A1_clara`↔`A1_ruido` puede bajar aunque sea la misma persona.
La magnitud depende del ruido, la reverberación, el micrófono y la calidad de la
grabación. La similitud coseno es un score, no una probabilidad ni una decisión
autónoma de identidad.


## Interpretación de resultados (verificación de locutor con ECAPA-TDNN)
* **Misma persona, distinta frase (`A1_clara.wav` vs. `A2_otra_frase.wav`)**: similitud de `[valor]` — al cambiar el texto pero mantener la misma voz, el embedding de ECAPA-TDNN conserva una similitud alta porque prioriza características de identidad del locutor frente al contenido léxico.
* **Misma persona, con ruido (`A1_clara.wav` vs. `A1_ruido.wav`)**: similitud de `[valor]` — el ruido degrada la representación acústica y reduce la similitud respecto a la señal limpia, aunque suele mantenerse por encima del umbral/comparación con otro locutor si el ruido no oculta por completo la voz.
* **Distinta persona (`A1_clara.wav` vs. `B1_otra_persona.wav`)**: similitud de `[valor]` — al cambiar de locutor, la distancia entre embeddings aumenta y la similitud cae respecto a las comparaciones del mismo hablante.